In [1]:
from Data_query.trino_config import *
import numpy as np
from visualisation import *
import pytz

In [11]:
stop_trino()

Trino service stopping triggered.


In [11]:
sleep(20)

In [2]:
big_workers = 1
workers = 0
num_workers = max(workers, big_workers)
ensure_trino_running(worker_desired_count = workers, big_worker_desired_count=big_workers)
# sleep(60)

Trino service is already running.


In [12]:
iceberg_sql(f""" select distinct site_id, ac_capacity_kw, state, min_time, max_time from meta_up23c 
            where is_pv=True and flex_export_detected=False and pf_01 > .98 and min_time < timestamp '2024-01-02'
            limit 2""")

,site_id,ac_capacity_kw,state,min_time,max_time
0,1233585204,5.0,VIC,2024-01-01,2024-06-04 20:55:00
1,158521185,8.0,NSW,2024-01-01,2025-06-06 07:10:00


In [ ]:
# Some sites are excluded because no CS day profile is detected for them. For example, site 1525233041, cs_day is 2024-01-21. But no ts data is detected for that day. 

In [3]:
iceberg_exec("drop table if exists pv_ghi_norm_model")
iceberg_exec(f"""
     create table pv_ghi_norm_model (
            site_id BIGINT,
            tod_bin TIME,
            a double,
            b double,
            n BIGINT
        )"""
            )
                     

Executed
Executed


In [8]:
sleep(20)

In [4]:
num_parts=1
time_bin_interval = '5' # in minutes
def run_func(args):
    year, month, part = args
    time_filter = f"year = {year}"
    part_filter = f"site_id % {num_parts} = {part}"
    df = iceberg_exec(f"""
                    insert into pv_ghi_norm_model
                    with train_val_data AS (
                        SELECT
                            site_id,
                            actual_day,
                            t_stamp,
                            CAST(
                                date_trunc('minute', t_stamp + interval '10' hour)
                                - interval '1' minute * (minute(t_stamp + interval '10' hour) % {time_bin_interval})
                                AS TIME) AS tod_bin,
                            GHI/GHI_cs AS x,
                            GHI_cs,
                            P_kw_norm/ NULLIF(P_kw_norm_cs, 0.0) AS y
                        FROM structured_data
                        WHERE P_kw_norm_cs > 0.2 AND GHI > 50 and P_kw_norm > 0.05 and P_kw_norm <= P_kw_norm_cs
                            and V <= 253 and (P_kw_norm >= 1 or S_norm < 1.001) and {time_filter} and {part_filter}
                    ),
                    train_data as (
                        SELECT t.*
                        FROM train_val_data t JOIN split_days s ON t.site_id = s.site_id AND t.actual_day = s.actual_day
                        WHERE s.day_type = 'train'),
                    model AS (
                        SELECT
                            site_id,
                            tod_bin,
                            (1 - regr_slope(y, x)) AS a,
                            regr_slope(y, x)     AS b,
                            count(*)             AS n
                        FROM train_data
                        GROUP BY site_id, tod_bin
                    )
                    select * from model
                    
    """)

    print(f"Completed {time_filter}, part {part}")
    return df
tasks = [(year, month, part) for year in (2024, ) for month in range(1, 2) 
         for part in range(0, num_parts)]
            
try:         
    res = trino_parallel(run_func, tasks, num_workers=num_workers)
except Exception as e:
    print(f"Error during data retrieval: {e}")
finally:
    # stop_trino()
    pass
res

Executed
Completed year = 2024, part 0


In [8]:
365-52*2

261

In [5]:
iceberg_sql(f"""select count(distinct site_id) from pv_ghi_norm_model
            """)

,_col0
0,13937
